# 01 — Data Cleaning, Duplicate ve Lipinski Analizi

Bu v5 sürümünde bütün kod hücreleri eğitim amacıyla satır satır açıklanmıştır. Her çalıştırılabilir satırın hemen üzerinde, o satırın görevini özetleyen kısa bir `#` notu bulunur.

Bu notebook yalnızca **kimyasal veri temizleme** görevini yapar.

- Notebook 00 çıktısını önce Google Drive'dan okur.
- Drive dosyası yoksa GitHub RAW bağlantısını dener.
- Tuzlar ve karşı iyonlar çıkarılır.
- Canonical parent SMILES oluşturulur.
- Duplicate yapılar medyan `pStandard` ile birleştirilir.
- Lipinski descriptorları ve geçiş durumu hesaplanır.

Pipeline'ın sonraki adımı **Lipinski ile filtrelenmemiş temiz CSV'yi** kullanır.
Lipinski filtreli CSV yalnızca karşılaştırma ve raporlama amacıyla kaydedilir.

## 1 — Paketler

Bu bölüm, notebookun ihtiyaç duyduğu Python paketlerini kontrol eder. Eksik paketler yalnızca gerektiğinde kurulur; böylece aynı kurulum komutları gereksiz yere tekrar çalıştırılmaz. Hücre tamamlandığında sonraki bölümlerde kullanılacak temel yazılımlar hazır olur.

In [ ]:
# importlib.util paketini kullanılabilir hâle getirir.
import importlib.util
# subprocess paketini kullanılabilir hâle getirir.
import subprocess
# sys paketini kullanılabilir hâle getirir.
import sys

# `PACKAGES` değişkenine bu adımda kullanılacak değeri atar.
PACKAGES = [
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("numpy", "numpy"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("pandas", "pandas"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("requests", "requests"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("rdkit", "rdkit"),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# Belirtilen koleksiyondaki öğeleri sırayla işler.
for import_name, pip_name in PACKAGES:
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if importlib.util.find_spec(import_name) is None:
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        subprocess.check_call(
            # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
            [sys.executable, "-m", "pip", "install", "-q", pip_name]
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Paket kontrolü tamamlandı.")


## 2 — Kütüphaneler ve pipeline ayarları

Bu bölümde kullanılacak kütüphaneler belleğe alınır ve pipeline boyunca değişmeden kullanılacak temel ayarlar tanımlanır. Target ID, klasör yolları, dosya adları ve tekrar üretilebilirlik değerleri burada merkezi olarak tutulur.

In [ ]:
# pathlib modülünden Path bileşenlerini içe aktarır.
from pathlib import Path
# io modülünden BytesIO bileşenlerini içe aktarır.
from io import BytesIO
# json paketini kullanılabilir hâle getirir.
import json

# numpy as np paketini kullanılabilir hâle getirir.
import numpy as np
# pandas as pd paketini kullanılabilir hâle getirir.
import pandas as pd
# requests paketini kullanılabilir hâle getirir.
import requests

# google.colab modülünden drive bileşenlerini içe aktarır.
from google.colab import drive
# IPython.display modülünden display bileşenlerini içe aktarır.
from IPython.display import display

# rdkit modülünden Chem bileşenlerini içe aktarır.
from rdkit import Chem
# rdkit.Chem modülünden Crippen, Descriptors, Lipinski bileşenlerini içe aktarır.
from rdkit.Chem import Crippen, Descriptors, Lipinski
# rdkit.Chem.MolStandardize modülünden rdMolStandardize bileşenlerini içe aktarır.
from rdkit.Chem.MolStandardize import rdMolStandardize

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
drive.mount("/content/drive", force_remount=False)

# `DRIVE_DIR` değişkenine bu adımda kullanılacak değeri atar.
DRIVE_DIR = Path(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "/content/drive/MyDrive/MOL_FET_regression_pipeline"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# `CONFIG_PATH` değişkenine bu adımda kullanılacak değeri atar.
CONFIG_PATH = DRIVE_DIR / "pipeline_config.json"

# `DEFAULT_TARGET_ID` değişkenine bu adımda kullanılacak değeri atar.
DEFAULT_TARGET_ID = "CHEMBL206"
# `DEFAULT_GITHUB_RAW_BASE` değişkenine bu adımda kullanılacak değeri atar.
DEFAULT_GITHUB_RAW_BASE = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "https://raw.githubusercontent.com/"
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "MOL-FEST/MOL-FET_regression/main"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if CONFIG_PATH.exists():
    # JSON metnini Python sözlüğüne dönüştürür.
    config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
# Önceki koşullar sağlanmadığında alternatif kod bloğunu çalıştırır.
else:
    # `config` değişkenine bu adımda kullanılacak değeri atar.
    config = {
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "target_id": DEFAULT_TARGET_ID,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "github_raw_base": DEFAULT_GITHUB_RAW_BASE,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "raw_filename": f"{DEFAULT_TARGET_ID}_IC50_single_protein_raw.csv",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "clean_unfiltered_filename": f"{DEFAULT_TARGET_ID}_clean_unfiltered.csv",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "lipinski_filtered_filename": f"{DEFAULT_TARGET_ID}_Lipinski_filtered.csv",
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    }

# `TARGET_ID` değişkenine bu adımda kullanılacak değeri atar.
TARGET_ID = config["target_id"]
# `GITHUB_RAW_BASE` değişkenine bu adımda kullanılacak değeri atar.
GITHUB_RAW_BASE = config["github_raw_base"]
# `RAW_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
RAW_FILENAME = config["raw_filename"]
# `CLEAN_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
CLEAN_FILENAME = config["clean_unfiltered_filename"]
# `LIPINSKI_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
LIPINSKI_FILENAME = config["lipinski_filtered_filename"]

# Gerekli klasörü mevcut değilse oluşturur.
DRIVE_DIR.mkdir(parents=True, exist_ok=True)


## 3 — Drive öncelikli CSV okuma

Bu bölüm veri kaynağını hazırlar. Pipeline önce Google Drive içindeki önceki notebook çıktısını arar; dosya bulunamazsa aynı dosyayı GitHub RAW bağlantısından okumayı dener. Böylece notebooklar hem ardışık Colab çalışmasına hem de GitHub üzerinden bağımsız kullanıma uygundur.

In [ ]:
# `load_csv` adlı yardımcı fonksiyonu tanımlar.
def load_csv(drive_filename, github_relative_path):
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    """CSV'yi önce Drive'dan, bulunamazsa GitHub'dan okur."""
    # `drive_path` değişkenine bu adımda kullanılacak değeri atar.
    drive_path = DRIVE_DIR / drive_filename

    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if drive_path.exists():
        # İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
        print("Girdi kaynağı: Google Drive")
        # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
        return pd.read_csv(drive_path, low_memory=False)

    # `github_url` değişkenine bu adımda kullanılacak değeri atar.
    github_url = f"{GITHUB_RAW_BASE}/{github_relative_path}"
    # Belirtilen internet adresine HTTP GET isteği gönderir.
    response = requests.get(github_url, timeout=120)
    # HTTP yanıtında hata kodu varsa işlemi açıklayıcı hatayla durdurur.
    response.raise_for_status()

    # İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
    print("Girdi kaynağı: GitHub")
    # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
    return pd.read_csv(BytesIO(response.content), low_memory=False)


# `raw_df` değişkenine bu adımda kullanılacak değeri atar.
raw_df = load_csv(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    RAW_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    f"data/{RAW_FILENAME}",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `required` değişkenine bu adımda kullanılacak değeri atar.
required = {"molecule_chembl_id", "canonical_smiles", "pStandard"}
# `missing` değişkenine bu adımda kullanılacak değeri atar.
missing = required.difference(raw_df.columns)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if missing:
    # Koşul sağlanmadığında açıklayıcı bir hata oluşturarak işlemi durdurur.
    raise KeyError(f"Eksik sütunlar: {sorted(missing)}")

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Ham veri:", raw_df.shape)


## 4 — Parent SMILES standardizasyonu

Bu bölüm moleküler yapıların kimyasal olarak tutarlı hâle getirilmesini sağlar. Geçersiz SMILES kayıtları ayrılır, tuz ve karşı iyonlar çıkarılır ve aynı parent yapıya ait tekrar ölçümler tek bir temsil altında birleştirilir.

In [ ]:
# `standardize_parent` adlı yardımcı fonksiyonu tanımlar.
def standardize_parent(smiles):
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    """Tuzları çıkarır ve canonical parent SMILES döndürür."""
    # Hata oluşturabilecek işlemleri güvenli bir blok içinde dener.
    try:
        # `molecule` değişkenine bu adımda kullanılacak değeri atar.
        molecule = Chem.MolFromSmiles(str(smiles))
        # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
        if molecule is None:
            # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
            return None

        # `parent` değişkenine bu adımda kullanılacak değeri atar.
        parent = rdMolStandardize.FragmentParent(molecule)
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        Chem.SanitizeMol(parent)

        # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
        return Chem.MolToSmiles(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            parent,
            # `canonical` değişkenine bu adımda kullanılacak değeri atar.
            canonical=True,
            # `isomericSmiles` değişkenine bu adımda kullanılacak değeri atar.
            isomericSmiles=True,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )
    # Belirtilen hata oluşursa kontrollü alternatif işlemi çalıştırır.
    except Exception:
        # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
        return None


# Orijinal veriyi değiştirmemek için bağımsız bir kopya oluşturur.
work_df = raw_df.copy()
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
work_df["pStandard"] = pd.to_numeric(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    work_df["pStandard"],
    # `errors` değişkenine bu adımda kullanılacak değeri atar.
    errors="coerce",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# Belirtilen fonksiyonu tablonun ilgili satır veya sütunlarına uygular.
work_df["parent_smiles"] = work_df["canonical_smiles"].apply(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    standardize_parent
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `invalid_df` değişkenine bu adımda kullanılacak değeri atar.
invalid_df = work_df.loc[
    # Eksik veya geçerli değerleri belirlemek için mantıksal maske üretir.
    work_df["pStandard"].isna()
    # Eksik veya geçerli değerleri belirlemek için mantıksal maske üretir.
    | work_df["parent_smiles"].isna()
# Orijinal veriyi değiştirmemek için bağımsız bir kopya oluşturur.
].copy()

# Eksik kritik değerleri içeren kayıtları çıkarır.
valid_df = work_df.dropna(
    # `subset` değişkenine bu adımda kullanılacak değeri atar.
    subset=["pStandard", "parent_smiles"]
# Orijinal veriyi değiştirmemek için bağımsız bir kopya oluşturur.
).copy()

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if valid_df.empty:
    # Koşul sağlanmadığında açıklayıcı bir hata oluşturarak işlemi durdurur.
    raise ValueError("Temizlik sonrasında geçerli molekül kalmadı.")

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Geçerli kayıt:", len(valid_df))
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Geçersiz kayıt:", len(invalid_df))


## 5 — Duplicate kayıtların birleştirilmesi

Bu bölüm moleküler yapıların kimyasal olarak tutarlı hâle getirilmesini sağlar. Geçersiz SMILES kayıtları ayrılır, tuz ve karşı iyonlar çıkarılır ve aynı parent yapıya ait tekrar ölçümler tek bir temsil altında birleştirilir.

In [ ]:
# `join_unique` adlı yardımcı fonksiyonu tanımlar.
def join_unique(values):
    # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
    return ";".join(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        sorted({str(value) for value in values if pd.notna(value)})
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )


# `clean_df` değişkenine bu adımda kullanılacak değeri atar.
clean_df = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    valid_df
    # Kayıtları belirtilen anahtar sütuna göre gruplandırır.
    .groupby("parent_smiles", as_index=False)
    # Gruplar üzerinde özet istatistikleri hesaplar.
    .agg(
        # `parent_chembl_id` değişkenine bu adımda kullanılacak değeri atar.
        parent_chembl_id=("molecule_chembl_id", join_unique),
        # `pStandard` değişkenine bu adımda kullanılacak değeri atar.
        pStandard=("pStandard", "median"),
        # `pStandard_mean` değişkenine bu adımda kullanılacak değeri atar.
        pStandard_mean=("pStandard", "mean"),
        # `pStandard_std` değişkenine bu adımda kullanılacak değeri atar.
        pStandard_std=("pStandard", "std"),
        # `pStandard_min` değişkenine bu adımda kullanılacak değeri atar.
        pStandard_min=("pStandard", "min"),
        # `pStandard_max` değişkenine bu adımda kullanılacak değeri atar.
        pStandard_max=("pStandard", "max"),
        # `n_measurements` değişkenine bu adımda kullanılacak değeri atar.
        n_measurements=("pStandard", "size"),
        # `n_assays` değişkenine bu adımda kullanılacak değeri atar.
        n_assays=("assay_chembl_id", "nunique"),
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Eksik değerleri belirtilen yöntem veya değerle doldurur.
clean_df["pStandard_std"] = clean_df["pStandard_std"].fillna(0.0)
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
clean_df["pStandard_range"] = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    clean_df["pStandard_max"] - clean_df["pStandard_min"]
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
clean_df["activity_conflict"] = clean_df["pStandard_range"] > 1.0

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if not clean_df["parent_smiles"].is_unique:
    # Koşul sağlanmadığında açıklayıcı bir hata oluşturarak işlemi durdurur.
    raise AssertionError("Duplicate parent SMILES kaldı.")

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Benzersiz parent yapı:", len(clean_df))


## 6 — Lipinski descriptorları

Bu bölüm klasik Lipinski özelliklerini hesaplar ve sonuçları raporlar. Lipinski filtresi pipeline'ın modelleme girdisini daraltmak için kullanılmaz; filtrelenmiş ve reddedilmiş moleküller yalnızca karşılaştırmalı inceleme amacıyla ayrı dosyalara yazılır.

In [ ]:
# `lipinski_values` adlı yardımcı fonksiyonu tanımlar.
def lipinski_values(smiles):
    # `molecule` değişkenine bu adımda kullanılacak değeri atar.
    molecule = Chem.MolFromSmiles(smiles)

    # `molecular_weight` değişkenine bu adımda kullanılacak değeri atar.
    molecular_weight = Descriptors.MolWt(molecule)
    # `logp` değişkenine bu adımda kullanılacak değeri atar.
    logp = Crippen.MolLogP(molecule)
    # `hbd` değişkenine bu adımda kullanılacak değeri atar.
    hbd = Lipinski.NumHDonors(molecule)
    # `hba` değişkenine bu adımda kullanılacak değeri atar.
    hba = Lipinski.NumHAcceptors(molecule)

    # `violations` değişkenine bu adımda kullanılacak değeri atar.
    violations = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        int(molecular_weight > 500)
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        + int(logp > 5)
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        + int(hbd > 5)
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        + int(hba > 10)
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
    return {
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "MolWt": molecular_weight,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "MolLogP": logp,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "HBD": hbd,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "HBA": hba,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "Lipinski_violations": violations,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "passes_Lipinski": violations == 0,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    }


# `lipinski_df` değişkenine bu adımda kullanılacak değeri atar.
lipinski_df = pd.DataFrame(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    clean_df["parent_smiles"]
    # Belirtilen fonksiyonu tablonun ilgili satır veya sütunlarına uygular.
    .apply(lipinski_values)
    # Diziyi veya pandas nesnesini standart Python listesine dönüştürür.
    .tolist()
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `clean_df` değişkenine bu adımda kullanılacak değeri atar.
clean_df = pd.concat(
    # DataFrame indeksini yeniden düzenler.
    [clean_df.reset_index(drop=True), lipinski_df],
    # `axis` değişkenine bu adımda kullanılacak değeri atar.
    axis=1,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `lipinski_filtered_df` değişkenine bu adımda kullanılacak değeri atar.
lipinski_filtered_df = clean_df.loc[
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    clean_df["passes_Lipinski"]
# Orijinal veriyi değiştirmemek için bağımsız bir kopya oluşturur.
].copy()

# `lipinski_rejected_df` değişkenine bu adımda kullanılacak değeri atar.
lipinski_rejected_df = clean_df.loc[
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    ~clean_df["passes_Lipinski"]
# Orijinal veriyi değiştirmemek için bağımsız bir kopya oluşturur.
].copy()

# Tabloyu notebook içinde okunabilir biçimde gösterir.
display(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    clean_df["passes_Lipinski"]
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    .value_counts()
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    .rename_axis("passes_Lipinski")
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    .to_frame("molecule_count")
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)


## 7 — Google Drive kayıtları

Bu bölüm veri kaynağını hazırlar. Pipeline önce Google Drive içindeki önceki notebook çıktısını arar; dosya bulunamazsa aynı dosyayı GitHub RAW bağlantısından okumayı dener. Böylece notebooklar hem ardışık Colab çalışmasına hem de GitHub üzerinden bağımsız kullanıma uygundur.

In [ ]:
# `clean_path` değişkenine bu adımda kullanılacak değeri atar.
clean_path = DRIVE_DIR / CLEAN_FILENAME
# `lipinski_path` değişkenine bu adımda kullanılacak değeri atar.
lipinski_path = DRIVE_DIR / LIPINSKI_FILENAME
# `rejected_path` değişkenine bu adımda kullanılacak değeri atar.
rejected_path = DRIVE_DIR / f"{TARGET_ID}_Lipinski_rejected.csv"
# `invalid_path` değişkenine bu adımda kullanılacak değeri atar.
invalid_path = DRIVE_DIR / f"{TARGET_ID}_invalid_records.csv"
# `report_path` değişkenine bu adımda kullanılacak değeri atar.
report_path = DRIVE_DIR / f"{TARGET_ID}_cleaning_report.csv"

# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
clean_df.to_csv(clean_path, index=False)
# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
lipinski_filtered_df.to_csv(lipinski_path, index=False)
# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
lipinski_rejected_df.to_csv(rejected_path, index=False)
# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
invalid_df.to_csv(invalid_path, index=False)

# `report` değişkenine bu adımda kullanılacak değeri atar.
report = pd.DataFrame(
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    {
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "metric": [
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "raw_rows",
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "invalid_rows",
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "unique_clean_molecules",
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "lipinski_pass",
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "lipinski_rejected",
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ],
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "value": [
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            len(raw_df),
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            len(invalid_df),
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            len(clean_df),
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            len(lipinski_filtered_df),
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            len(lipinski_rejected_df),
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ],
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    }
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
report.to_csv(report_path, index=False)

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Pipeline girdisi olarak kullanılacak dosya:")
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print(clean_path)
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Lipinski filtreli dosya yalnızca rapor amaçlıdır:")
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print(lipinski_path)
